# 标准库容器与函数工具

学习目标：根据计数、排队、顺序查找和重复计算的需要选择标准库工具，并识别惰性迭代与缓存的状态边界。

前置知识：列表与字典、可哈希性、对象引用、函数参数、迭代器与生成器、装饰器、类和异常处理。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 Counter 的计数与增减

### 1.1 从记录得到频次

collections 提供有专门用途的容器；本章在内置容器与迭代协议的基础上，关注工具选择和行为边界。

Counter 是 dict 的子类，以可哈希对象为键、出现次数为值。读取不存在的键得到 0，不必预先为每个对象建键；update 累加计数，不像 dict.update 那样覆盖原值。

most\_common 返回按计数从多到少排列的条目；传入整数 2 就取前两项。计数相同时按首次出现顺序排列。total 返回全部计数之和。

In [1]:
import collections

courses = ["python", "sql", "python", "web", "sql", "python"]
counts = collections.Counter(courses)
print(counts.most_common(2))  # [('python', 3), ('sql', 2)]
print(counts["rust"], "rust" in counts)  # 0 False：读取没有新增键。

counts.update(["sql", "sql"])
print(counts["sql"], counts.total())  # 4 8：在原计数上累加。

[('python', 3), ('sql', 2)]
0 False
4 8


### 1.2 零和负数不会自动删除

Counter 允许零与负计数。subtract 原地扣减，并保留这些结果；把某项设为 0 也不会删除键。

相减运算与 subtract 不同：两个 Counter 相减得到新 Counter，只保留正计数。+balance 中的 balance 表示一个 Counter，一元加号同样返回只含正计数的新对象。

In [2]:
balance = collections.Counter({"python": 3, "sql": 1})
used = collections.Counter({"python": 1, "sql": 2, "web": 1})
print(dict(balance - used))  # {'python': 2}：运算结果只含正计数。

balance.subtract(used)
balance["rust"] = 0
print(dict(balance))  # python 为 2，sql、web 为 -1，rust 为 0。
print("rust" in balance)  # True：零计数仍然占有一个键。
print(dict(+balance))  # {'python': 2}：原 balance 未被过滤。
del balance["rust"]
assert "rust" not in balance

{'python': 2}
{'python': 2, 'sql': -1, 'web': -1, 'rust': 0}
True
{'python': 2}


## 2 defaultdict 为缺失键创建容器

defaultdict 接收一个无参数工厂，例如 list，而不是提前调用得到的 list()。使用方括号读取缺失键时，工厂创建默认值，字典保存它，再把它返回。

这适合按课程收集多次学习时长，每个新课程得到独立列表。get 和成员检查不触发工厂；若只想查看键是否存在，不应通过方括号顺便创建它。

In [3]:
records = [("python", 30), ("sql", 20), ("python", 45)]
minutes_by_course = collections.defaultdict(list)
for course, minutes in records:
    minutes_by_course[course].append(minutes)

print(dict(minutes_by_course))  # {'python': [30, 45], 'sql': [20]}
print(minutes_by_course.get("web"))  # None：没有调用 list 工厂。
print("web" in minutes_by_course)  # False
print(minutes_by_course["web"])  # []：这次读取创建并保存新列表。
print("web" in minutes_by_course)  # True
assert minutes_by_course["python"] is not minutes_by_course["sql"]

{'python': [30, 45], 'sql': [20]}
None
False
[]
True


## 3 deque 的两端操作与容量

deque 是双端队列（double-ended queue），适合在两端追加和取出元素。append 在右端追加，appendleft 在左端追加；pop 从右端取出，popleft 从左端取出。空队列取出会抛出 IndexError。

两端追加和取出的成本约为 O(1)，即不随元素数线性增长；中间索引访问则为 O(n)，n 表示队列中的元素数。需要频繁随机索引时，list 更合适。

In [4]:
pending = collections.deque(["读取", "解析"])
pending.append("保存")
pending.appendleft("校验")
print(pending.popleft())  # 校验：从左端开始处理。
print(pending.pop())  # 保存：也可以操作右端。
print(list(pending))  # ['读取', '解析']

empty_queue = collections.deque()

校验
保存
['读取', '解析']


In [5]:
# 预期 IndexError：直接观察原始异常，之后继续运行下一单元。
# IndexError：队列为空。
empty_queue.popleft()

IndexError: pop from an empty deque

### 3.1 有界队列适合保留最近记录

maxlen 指定最大长度。队列已满时，append 向右追加会丢弃最左端的旧项，appendleft 则相反；这适合保留最近记录，不适合要求每项都必须处理的待办队列。

extendleft 逐个向左追加，因此输入顺序会反转。rotate(1) 把最右项移到最左，负数则向左轮转。

In [6]:
recent = collections.deque(["读取", "解析"], maxlen=2)
recent.append("保存")
print(list(recent))  # ['解析', '保存']：最旧的“读取”被丢弃。

queue = collections.deque(["原有"])
queue.extendleft(["先", "后"])
print(list(queue))  # ['后', '先', '原有']：逐个向左追加。
queue.rotate(1)
print(list(queue))  # ['原有', '后', '先']

['解析', '保存']
['后', '先', '原有']
['原有', '后', '先']


## 4 namedtuple 为元组字段命名

namedtuple 是创建元组子类的工厂，适合字段固定、需要按名称读取的小型记录。下面的 Study 保存 course（课程名）和 minutes（学习分钟数），实例同时支持属性访问、索引和解包。

字段不能重新赋值；\_replace 返回替换字段后的新实例，\_asdict 返回按字段名组织的新字典。这两个带单下划线的方法是 namedtuple 的公开接口。

In [7]:
Study = collections.namedtuple("Study", ["course", "minutes"])
Study.__doc__ = "记录一次课程学习及其分钟数。"

study = Study("python", 30)
revised = study._replace(minutes=45)
print(study.course, study[1])  # python 30
print(revised._asdict())  # {'course': 'python', 'minutes': 45}
print(study.minutes)  # 30：替换没有修改原记录。

python 30
{'course': 'python', 'minutes': 45}
30


In [8]:
# 预期 AttributeError：直接观察原始异常，之后继续运行下一单元。
# AttributeError：元组字段不能重新赋值。
study.minutes = 60

AttributeError: can't set attribute

## 5 ChainMap 按优先级读取配置

ChainMap 把多个映射连接为一个视图。查找按传入顺序进行，先找到的值优先；赋值和删除只作用于第一个映射。

它保留原映射的引用，不复制底层数据。new\_child 可在前面增加一层局部配置，使这层的赋值不改动已有映射。

In [9]:
defaults = {"format": "text", "limit": 10}
overrides = {"format": "json"}
settings = collections.ChainMap(overrides, defaults)
print(settings["format"], settings["limit"])  # json 10

settings["limit"] = 3
print(overrides, defaults["limit"])  # 首层新增 limit=3，默认值仍为 10。
del settings["limit"]
defaults["limit"] = 20
print(settings["limit"])  # 20：删除首层覆盖后，能看到底层的新值。

local_settings = settings.new_child({"format": "csv"})
print(local_settings["format"], settings["format"])  # csv json

json 10
{'format': 'json', 'limit': 3} 10
20
csv json


## 6 heapq 总是先取最小项

### 6.1 最小堆不等于排序列表

最小堆（min-heap）要求每个父节点不大于子节点，因此根节点是最小项；它不要求整个列表从左到右有序。Python 3.12 的 heapq 用列表保存最小堆，索引 0 对应根节点。

heapify 原地把列表整理成堆，heappush 加入元素并维持堆条件，heappop 取出最小元素并维持堆条件。需要升序结果时，可以不断弹出；不能直接把堆的内部排列当成排序结果。

In [10]:
import heapq

priorities = [9, 2, 7, 4, 1, 6]
heapq.heapify(priorities)
print(priorities[0])  # 1：最小值位于根节点。
print(priorities == sorted(priorities))  # 本例为 False：堆并非整体有序。

heapq.heappush(priorities, 3)
ordered_priorities = []
while priorities:
    ordered_priorities.append(heapq.heappop(priorities))
print(ordered_priorities)  # [1, 2, 3, 4, 6, 7, 9]
assert not priorities  # 弹出会消耗原堆；空堆继续 heappop 会抛 IndexError。

1
False
[1, 2, 3, 4, 6, 7, 9]


### 6.2 同优先级任务增加顺序号

把优先级与任务放进元组时，优先级相同就会继续比较后续字段。任务本身可能没有大小关系，例如字典。

可以保存三项：优先级、递增顺序号、任务。顺序号唯一，使相同优先级按加入顺序取出，同时避免比较任务对象。本例约定优先级数值越小越先处理。

In [11]:
tasks = [
    (2, {"name": "整理"}),
    (1, {"name": "复习"}),
    (1, {"name": "练习"}),
]
task_heap = []
for sequence, (priority, task) in enumerate(tasks):
    heapq.heappush(task_heap, (priority, sequence, task))

while task_heap:
    priority, sequence, task = heapq.heappop(task_heap)
    print(task["name"])
# 复习、练习、整理：相同优先级保持加入顺序，未比较字典。

复习
练习
整理


## 7 bisect 查找有序列表的插入点

### 7.1 左边界与右边界

bisect 要求输入已经按相同规则升序排列，它不会先替你排序。返回值是插入位置，即使待查值不存在也会返回一个位置；不能只凭位置判断找到了该值。

| API | 中文名称／含义 |
| --- | --- |
| bisect.bisect\_left | 找到现有相等项之前的插入点 |
| bisect.bisect\_right | 找到现有相等项之后的插入点 |
| bisect.insort\_left | 在左插入点插入新项，保持有序 |
| bisect.insort\_right | 在右插入点插入新项，保持有序 |

下面用左右插入点之差统计重复值，并在确认位置未越界后检查值是否相等。

In [12]:
import bisect

durations = [10, 20, 20, 40]
left = bisect.bisect_left(durations, 20)
right = bisect.bisect_right(durations, 20)
print(left, right, right - left)  # 1 3 2：两个 20 位于左右边界之间。

for target in (20, 30, 50):
    position = bisect.bisect_left(durations, target)
    found = position < len(durations) and durations[position] == target
    print(target, position, found)
# 20 1 True；30 3 False；50 4 False：插入点可能等于列表长度。

1 3 2
20 1 True
30 3 False
50 4 False


### 7.2 二分查找快，列表插入仍需移动元素

设 n 为列表长度。查找插入点为 O(log n)，但列表中间插入需要移动元素，insort 的总体成本为 O(n)，不能把查找成本当成整个插入成本。

按记录字段排序时可传 key 函数。bisect 的 key 只用于已有记录，待查参数应直接提供比较键；insort 则接收完整新记录，在查找时也会计算新记录的 key。

In [13]:
schedule = [(20, "已有甲"), (20, "已有乙"), (40, "复习")]
left_schedule = schedule.copy()
right_schedule = schedule.copy()

# 本例 key 提取元组首项，即分钟数；查找传 20，插入传完整记录。
print(bisect.bisect_left(schedule, 20, key=lambda item: item[0]))  # 0
bisect.insort_left(
    left_schedule, (20, "新增"), key=lambda item: item[0]
)
bisect.insort_right(
    right_schedule, (20, "新增"), key=lambda item: item[0]
)
print([name for minutes, name in left_schedule])
print([name for minutes, name in right_schedule])
# 左插入：新增、已有甲、已有乙、复习。
# 右插入：已有甲、已有乙、新增、复习。

0
['新增', '已有甲', '已有乙', '复习']
['已有甲', '已有乙', '新增', '复习']


## 8 chain、islice 与有界消费

itertools 的这些工具返回迭代器，按消费进度处理数据。chain 依次读取多个输入，不先拼出完整列表；islice 从迭代器中选择一段位置，也会消耗被跳过的元素。

count 从 start 开始，每次增加 step，默认分别为 0 和 1；它没有自然终点。应通过有限的 islice 等方式限定消费，不能直接对 count 调用 list。

islice 的起止位置不能为负数，步长必须为正数；stop 是不包含的右端位置。下面只从无限序列中读取四项。

In [14]:
import itertools

combined = itertools.chain(["读取", "解析"], ["保存"])
print(list(itertools.islice(combined, 2)))  # ['读取', '解析']
print(list(combined))  # ['保存']：前两项已经被消费。
print(list(combined))  # []：迭代器已耗尽。

numbers = itertools.count(start=10, step=5)
print(list(itertools.islice(numbers, 4)))  # [10, 15, 20, 25]：有限消费。

source = iter([10, 20, 30, 40, 50])
print(list(itertools.islice(source, 1, 3)))  # [20, 30]：10 也被跳过并消费。
print(next(source))  # 40：不是从原输入重新开始。

['读取', '解析']
['保存']
[]
[10, 15, 20, 25]
[20, 30]
40


## 9 groupby 只合并相邻同键项

### 9.1 先确定是在找连续段，还是汇总同类记录

groupby 在键值改变时开始新组，相同键隔开后会再次出现。若要把全部同类项放在一组，先用同一个 key 排序。

这与 SQL 的 GROUP BY 不同，后者不要求同组记录在输入中相邻。下面直接分组得到连续课程段，排序后分组才得到每种课程的完整记录。

In [15]:
sessions = [("python", 30), ("sql", 20), ("python", 45)]
for course, group in itertools.groupby(sessions, key=lambda item: item[0]):
    print(course, list(group))
# 依次为 python [('python', 30)]、sql [('sql', 20)]、python [('python', 45)]。

ordered_sessions = sorted(sessions, key=lambda item: item[0])
for course, group in itertools.groupby(
    ordered_sessions, key=lambda item: item[0]
):
    print(course, sum(minutes for name, minutes in group))
# python 75；sql 20：排序后相同课程连续，才能在这里一次汇总完整。

python [('python', 30)]
sql [('sql', 20)]
python [('python', 45)]
python 75
sql 20


### 9.2 分组迭代器共享底层输入

每个组本身也是迭代器，与外层 groupby 共用输入。外层移动到下一组后，上一组尚未消费的内容不再可用。

需要稍后使用分组内容时，应在推进外层之前把当前组转为 list；只保存分组迭代器并不能保存该组数据。

In [16]:
groups = itertools.groupby(["python", "python", "sql"])
first_key, first_group = next(groups)
second_key, second_group = next(groups)
print(first_key, list(first_group))  # python []：外层已越过第一组。
print(second_key, list(second_group))  # sql ['sql']

saved_groups = []
for course, group in itertools.groupby(["python", "python", "sql"]):
    saved_groups.append((course, list(group)))
print(saved_groups)
# [('python', ['python', 'python']), ('sql', ['sql'])]：及时保存了各组。

python []
sql ['sql']
[('python', ['python', 'python']), ('sql', ['sql'])]


## 10 tee 的独立进度需要缓冲

tee 从一个输入产生消费进度独立的迭代器，不会重新运行输入生成过程。快支读过、慢支尚未读取的值需要暂存，因此进度差越大，所需额外存储也可能越大。

![tee 共享输入，分别记录消费进度](image/illustration/21-01-tee-buffer.svg)

图示：tee 的进度差与逻辑缓冲。输入只向前读取一次，落后的分支从已保存值继续。

分流后只使用返回的分支，不另行推进原迭代器；tee 分支也不保证线程安全。若一支几乎读完才开始另一支，直接保存 list 往往更合适。

下面的“读取”标记只在源生成器推进时出现。将慢支首次取得 10 与标记对照，确认它读取的是缓冲；本图与实验均不推断具体内存字节数。

In [17]:
from collections.abc import Iterator


def iter_minutes() -> Iterator[int]:
    """产出三条分钟数，并显示源数据何时被读取。"""
    for minutes in (10, 20, 30):
        print("读取", minutes)
        yield minutes


fast, slow = itertools.tee(iter_minutes())
print("快支", next(fast), next(fast))  # 先打印“读取 10”“读取 20”，再打印“快支 10 20”。
print("慢支", next(slow))  # 得到 10，没有再次打印“读取 10”。
print("快支剩余", list(fast))  # 读取 30，得到 [30]，并耗尽输入。
print("慢支剩余", list(slow))  # [20, 30]：来自缓冲，两支都完成消费。

读取 10
读取 20
快支 10 20
慢支 10
读取 30
快支剩余 [30]
慢支剩余 [20, 30]


## 11 partial 预设部分参数

functools.partial 接收一个可调用对象和预设参数，返回新的可调用对象；创建时不会提前执行原函数。

调用新对象时，新增的位置参数接在预设位置参数后面，新增的关键字参数补充或覆盖预设关键字参数。partial 适合固定重复配置，但预设关键字不代表调用者不能覆盖它。

In [18]:
import functools


def format_study(course: str, minutes: int, *, unit: str = "分钟") -> str:
    """按课程名、时长和单位生成一行记录。"""
    return f"{course}：{minutes} {unit}"


format_python = functools.partial(format_study, "python", unit="分钟")
print(format_python(30))  # python：30 分钟
print(format_python(2, unit="小时"))  # python：2 小时，覆盖预设关键字。
assert format_python.func is format_study

python：30 分钟
python：2 小时


## 12 缓存的容量、引用与失效

### 12.1 用 lru\_cache 限制条目数量

函数缓存根据调用参数复用已计算结果。lru\_cache 采用最近最少使用（least recently used，LRU）规则；容量满时淘汰最久未使用的条目。maxsize 默认 128，下面显式设为 2。

cache 没有容量上限，等价于 lru\_cache(maxsize=None)。容量限制的是条目数，不是总字节数；缓存会保留参数和返回值的引用，直到条目被淘汰或清空。

| API／字段 | 中文名称／含义 |
| --- | --- |
| cache\_info() | 返回命中、未命中和容量统计 |
| hits | 命中次数，即复用了已有结果 |
| misses | 未命中次数，即需要计算结果 |
| maxsize | 缓存条目容量上限，None 表示无上限 |
| currsize | 当前缓存条目数 |
| cache\_clear() | 清空条目并重置统计 |

本例的计算很小，只观察命中和淘汰规则，不据此宣称缓存提升了性能。

In [19]:
@functools.lru_cache(maxsize=2)
def total_minutes(durations: tuple[int, ...]) -> int:
    """汇总不可变的时长序列。"""
    return sum(durations)


for durations in ((10, 20), (5,), (10, 20), (8,), (5,)):
    print(total_minutes(durations))
# 30、5、30、8、5；第二次 (10, 20) 命中。
# 插入 (8,) 时淘汰较久未用的 (5,)，最后一次 (5,) 需重新计算。
info = total_minutes.cache_info()
print(info.hits, info.misses, info.maxsize, info.currsize)  # 1 4 2 2
total_minutes.cache_clear()
print(total_minutes.cache_info().currsize)  # 0：引用和统计不无限保留。

30
5
30
8
5
1 4 2 2
0


### 12.2 参数必须可哈希

缓存用字典保存调用结果，因此位置和关键字参数都必须可哈希。list 不能作为缓存键；若任务本来就只需要不变的值序列，可以显式转换为元素可哈希的 tuple。

缓存按调用参数建条目，不会自动把所有等价调用归一化；关键字顺序不同，也可能形成不同条目。不要假设“业务含义相同”就一定命中。

In [20]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
# TypeError：列表不可哈希，尚未进入原函数。
total_minutes([10, 20])

TypeError: unhashable type: 'list'

In [21]:
print(total_minutes(tuple([10, 20])))  # 30：明确转换为不可变输入。
total_minutes.cache_clear()

30


### 12.3 可变返回值会被下一次调用复用

缓存保留返回对象的引用，不会为每次命中复制对象。若调用者修改了缓存中的列表，后续命中就会看到这次修改。

需要每次获得独立可变对象的函数不应直接缓存。下面先观察反例，再让缓存返回元组，并由调用者在缓存外创建自己的列表。

In [22]:
@functools.cache
def split_courses_bad(text: str) -> list[str]:
    """反例：缓存可变列表，调用者会共享同一个结果。"""
    return text.split(",")


first_courses = split_courses_bad("python,sql")
first_courses.append("web")
print(split_courses_bad("python,sql"))  # ['python', 'sql', 'web']
print(first_courses is split_courses_bad("python,sql"))  # True
split_courses_bad.cache_clear()


@functools.cache
def split_courses(text: str) -> tuple[str, ...]:
    """缓存不可变的课程名称序列。"""
    return tuple(text.split(","))


own_courses = list(split_courses("python,sql"))
own_courses.append("web")
print(split_courses("python,sql"))  # ('python', 'sql')：缓存对象未被修改。
split_courses.cache_clear()

['python', 'sql', 'web']
True
('python', 'sql')


### 12.4 对象状态变化不会自动使缓存失效

缓存方法时，self 也属于键；同一个实例的属性变化并不会自动使原条目失效。若返回值依赖可变状态，应在状态变化后显式清空，或把决定结果的值作为参数传入。

cache 和 lru\_cache 是当前进程中该包装对象的缓存，不是磁盘存储，也不会自动监测外部文件或数据库变化。它们适合复用稳定结果，不适合直接缓存有副作用、随机结果或需要重新创建生成器／协程的调用。

In [23]:
class StudyPlan:
    """反例：每日分钟数可变，但方法缓存不会自动跟随属性变化。"""

    def __init__(self, daily_minutes: int) -> None:
        self.daily_minutes = daily_minutes

    @functools.lru_cache(maxsize=4)
    def total(self, days: int) -> int:
        """根据当前每日分钟数计算总量；演示状态变化后的陈旧缓存。"""
        return self.daily_minutes * days


# 先展示可变实例状态导致旧结果命中，再对比把状态显式放入参数。
plan = StudyPlan(30)
print(plan.total(2))  # 60
plan.daily_minutes = 45
print(plan.total(2))  # 60：仍命中旧条目，并非新状态的计算结果。
StudyPlan.total.cache_clear()  # 清的是这个方法包装器的全部条目。
print(plan.total(2))  # 90：清空后才重新计算。
StudyPlan.total.cache_clear()


@functools.cache
def planned_minutes(daily_minutes: int, days: int) -> int:
    """把决定结果的两个值都放进缓存参数。"""
    return daily_minutes * days


print(planned_minutes(30, 2), planned_minutes(45, 2))  # 60 90
planned_minutes.cache_clear()

60
60
90
60 90


## 本章小结

（1）Counter 汇总频次，defaultdict 为缺失键创建值，deque 处理两端操作，namedtuple 为固定字段命名，ChainMap 按优先级读取多个映射。

（2）heapq 保证最小项在根部，不保证列表整体有序；bisect 需要已排序输入，查找插入点和移动元素的成本不同。

（3）itertools 按消费进度处理数据。无限输入必须限制消费；groupby 只合并相邻同键项，tee 的独立进度依赖缓冲。

（4）partial 预设参数；缓存复用结果，还会保留参数与结果引用。容量、可哈希性、可变返回值及状态失效需要一起考虑。

自查：能否说明一次“读取”是否会新增键、消耗输入，或返回与上次相同的可变对象？

## 练习

（1）先预测下面三行输出，再运行核对。分别解释 defaultdict.get 是否建键、bisect 返回值的含义，以及 groupby 为何得到这些组。核对标准是值与顺序均一致，且能说明把最后一行输入先排序后会怎样改变分组。

In [24]:
exercise_groups = collections.defaultdict(list)
print(exercise_groups.get("python"), "python" in exercise_groups)
print(bisect.bisect_left([10, 20, 20, 40], 20))
print([(key, list(group)) for key, group in itertools.groupby("AABA")])
# 先写预测，再运行；不要把插入位置当成查找成功标志。

None

 False
1
[('A', ['A', 'A']), ('B', ['B']), ('A', ['A'])]


（2）用 Counter 统计下面的任务名称，用 deque(maxlen=2) 保留最近两个名称，再用 heapq 按优先级取出任务；同优先级按输入顺序处理。

检查 review 出现两次、recent 保留 review 和 write，取出顺序为第二条 review、write、第一条 review。任务中加入顺序号以区分两次 review；处理完后堆应为空。

In [25]:
exercise_tasks = [(2, "review"), (1, "review"), (1, "write")]
# 在此统计名称、记录最近两项，并构建含顺序号的堆。
# 对照输入位置核对队列与堆的顺序，不仅比较名称集合。

（3）编写 parse\_minutes(text)，把逗号分隔的整数文本转换为元组，使用 lru\_cache(maxsize=2) 缓存。重复调用相同文本后检查一次未命中、一次命中；分别调用另两个文本后，检查当前条目数为 2。

让调用者通过 list 创建可修改副本，追加元素后缓存元组应保持原值。用 cache\_clear 清空后检查命中、未命中和条目数都归零。仅使用本题提供的有效整数文本，不增加未约定的错误回退。

In [26]:
exercise_text = "10,20"
# 在此定义带中文 docstring 的 parse_minutes，并核对缓存统计。
# 分别使用 "5"、"8" 填满并替换条目，再清空缓存。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [collections 与专用容器](https://docs.python.org/3.12/library/collections.html#module-collections)；[Counter 的缺失键、零计数与运算](https://docs.python.org/3.12/library/collections.html#collections.Counter)、[most\_common](https://docs.python.org/3.12/library/collections.html#collections.Counter.most_common)、[update](https://docs.python.org/3.12/library/collections.html#collections.Counter.update)、[subtract](https://docs.python.org/3.12/library/collections.html#collections.Counter.subtract)、[total](https://docs.python.org/3.12/library/collections.html#collections.Counter.total)；[defaultdict 工厂与 get 边界](https://docs.python.org/3.12/library/collections.html#collections.defaultdict)；[deque 的两端操作、复杂度与 maxlen](https://docs.python.org/3.12/library/collections.html#collections.deque)、[extendleft 的反序](https://docs.python.org/3.12/library/collections.html#collections.deque.extendleft)、[rotate](https://docs.python.org/3.12/library/collections.html#collections.deque.rotate)；[namedtuple](https://docs.python.org/3.12/library/collections.html#collections.namedtuple)、[字段替换](https://docs.python.org/3.12/library/collections.html#collections.somenamedtuple._replace)、[转换字典](https://docs.python.org/3.12/library/collections.html#collections.somenamedtuple._asdict)；[ChainMap 的查找、修改与引用](https://docs.python.org/3.12/library/collections.html#collections.ChainMap)、[new\_child](https://docs.python.org/3.12/library/collections.html#collections.ChainMap.new_child)；[最小堆条件](https://docs.python.org/3.12/library/heapq.html#module-heapq)、[heapify](https://docs.python.org/3.12/library/heapq.html#heapq.heapify)、[heappush](https://docs.python.org/3.12/library/heapq.html#heapq.heappush)、[heappop](https://docs.python.org/3.12/library/heapq.html#heapq.heappop)、[相同优先级的顺序号](https://docs.python.org/3.12/library/heapq.html#priority-queue-implementation-notes)；[左插入点](https://docs.python.org/3.12/library/bisect.html#bisect.bisect_left)、[右插入点](https://docs.python.org/3.12/library/bisect.html#bisect.bisect_right)、[左插入与 key](https://docs.python.org/3.12/library/bisect.html#bisect.insort_left)、[右插入](https://docs.python.org/3.12/library/bisect.html#bisect.insort_right)、[查找与插入成本](https://docs.python.org/3.12/library/bisect.html#performance-notes)；[chain](https://docs.python.org/3.12/library/itertools.html#itertools.chain)、[count](https://docs.python.org/3.12/library/itertools.html#itertools.count)、[islice](https://docs.python.org/3.12/library/itertools.html#itertools.islice)、[groupby 的相邻分组与共享输入](https://docs.python.org/3.12/library/itertools.html#itertools.groupby)、[tee 的缓冲与线程边界](https://docs.python.org/3.12/library/itertools.html#itertools.tee)；[partial 参数组合](https://docs.python.org/3.12/library/functools.html#functools.partial)、[cache 的无界缓存](https://docs.python.org/3.12/library/functools.html#functools.cache)、[lru\_cache 的参数、统计、引用与适用范围](https://docs.python.org/3.12/library/functools.html#functools.lru_cache)。 |
| GitHub 原始源码（CPython 3.12.14） | [缓存清理实现](https://raw.githubusercontent.com/python/cpython/v3.12.14/Modules/_functoolsmodule.c)，第 1318–1335 行的 cache\_clear 实现同时清空条目和重置统计。 |